In [1]:
!pip install transformers datasets seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from seqeval.metrics import classification_report

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

print("Path to dataset files:", path)

100%|██████████| 960k/960k [00:00<00:00, 121MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1


In [6]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1/test.txt
/root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1/metadata
/root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1/valid.txt
/root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1/train.txt


In [7]:
def read_conll(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        words = []
        tags = []

        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
            else:
                parts = line.strip().split()
                words.append(parts[0])
                tags.append(parts[-1])  # last column = tag

        if words:
            sentences.append(words)
            labels.append(tags)

    return sentences, labels

train_sentences, train_labels = read_conll(os.path.join(path, "train.txt"))
val_sentences, val_labels = read_conll(os.path.join(path, "valid.txt"))
test_sentences, test_labels = read_conll(os.path.join(path, "test.txt"))

print("Sample sentence:", train_sentences[0])
print("Sample labels:", train_labels[0])

Sample sentence: ['-DOCSTART-']
Sample labels: ['O']


In [8]:
unique_labels = set(label for sent in train_labels for label in sent)
label_list = list(unique_labels)

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)

['I-LOC', 'O', 'I-ORG', 'I-MISC', 'I-PER', 'B-ORG', 'B-MISC', 'B-PER', 'B-LOC']


In [9]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
def tokenize_and_align(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(label2id[label[word_idx]])

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [11]:
train_encodings = tokenize_and_align(train_sentences, train_labels)
val_encodings = tokenize_and_align(val_sentences, val_labels)

In [12]:
import torch

class NERDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = NERDataset(train_encodings)
val_dataset = NERDataset(val_encodings)

In [13]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [16]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.088790,0.067212
2,0.032068,0.057809


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3748, training_loss=0.08492126459754773, metrics={'train_runtime': 1433.3562, 'train_samples_per_second': 20.912, 'train_steps_per_second': 2.615, 'total_flos': 2508881178149136.0, 'train_loss': 0.08492126459754773, 'epoch': 2.0})

In [18]:
predictions, labels, _ = trainer.predict(val_dataset)

preds = np.argmax(predictions, axis=2)

true_labels = [
    [id2label[l] for l in label if l != -100]
    for label in labels
]

true_preds = [
    [id2label[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

print(classification_report(true_labels, true_preds))

              precision    recall  f1-score   support

         LOC       0.96      0.97      0.96      2618
        MISC       0.84      0.85      0.84      1231
         ORG       0.90      0.94      0.92      2056
         PER       0.98      0.97      0.97      3034

   micro avg       0.93      0.95      0.94      8939
   macro avg       0.92      0.93      0.92      8939
weighted avg       0.93      0.95      0.94      8939

